# 03 – Bronze load

8 ham Berka dosyası `workspace.bronze` altında Delta tablolara yüklenir. Tip dönüşümü yok: tüm kolonlar string kalır, iki metadata kolonu eklenir.

## Ayarlar ve okuma function'ı

Amaç: Ham dosyaları okuyan tek bir function yazmak ve her satıra iki metadata kolonu eklemek: `_ingest_ts` (yükleme zamanı) ve `_source_file` (satırın geldiği dosya).

`_source_file`, Databricks'in dosya kaynaklarında verdiği gizli `_metadata` kolonundan alınır.

Çıktı: `district`'ten 3 satır gelir. `_source_file`'da dosyanın tam path'i, `_ingest_ts`'de şu anki zaman görünmeli.

In [0]:
from pyspark.sql import functions as F

RAW_PATH = "/Volumes/workspace/bronze/raw/"
BRONZE_SCHEMA = "workspace.bronze"

# file name -> bronze table name ("order" is a SQL reserved word)
table_names = {
    "account": "account",
    "card": "card",
    "client": "client",
    "disp": "disp",
    "district": "district",
    "loan": "loan",
    "order": "orders",
    "trans": "trans",
}


def read_raw(file_name):
    path = RAW_PATH + file_name + ".csv"
    df_raw = spark.read.option("header", True).option("sep", ";").csv(path)
    df_with_ts = df_raw.withColumn("_ingest_ts", F.current_timestamp())
    df_with_meta = df_with_ts.withColumn("_source_file", F.col("_metadata.file_path"))
    return df_with_meta


df_test = read_raw("district")
display(df_test.select("A1", "A2", "_ingest_ts", "_source_file").limit(3))

## Bronze Delta tablolarına yazma

Amaç: 8 ham dosyayı okuyup `workspace.bronze` altında Delta tablolarına yazmak. Kolon isimleri değişmez; sadece `_ingest_ts` ve `_source_file` eklenir. `order` dosyası `orders` tablosuna yazılır (SQL reserved word).

Yazım modu `overwrite`: notebook tekrar çalıştırılırsa tablo yenilenir, duplicate oluşmaz. Delta her yazımda yeni bir version açar.

Çıktı: Her tablo için bir satır yazılır (`written: workspace.bronze.<tablo>`).

In [0]:
for file_name in table_names:
    table_name = table_names[file_name]
    full_table_name = BRONZE_SCHEMA + "." + table_name

    df_bronze = read_raw(file_name)

    (df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name))

    print("written:", full_table_name)

## Bronze satır sayıları

Amaç: Bronze tablolardaki satır sayılarının beklenen değerlerle (02_raw_data_check'te teyit edilen ham dosya sayıları) aynı olduğunu kontrol etmek.

Çıktı: Her tablo için `OK` ya da `MISMATCH` yazılır, en altta toplam gelir. Beklenen toplam: 1.079.680.

In [0]:
expected_counts = {
    "account": 4500,
    "card": 892,
    "client": 5369,
    "disp": 5369,
    "district": 77,
    "loan": 682,
    "orders": 6471,
    "trans": 1056320,
}

total_rows = 0

for table_name in expected_counts:
    full_table_name = BRONZE_SCHEMA + "." + table_name
    actual = spark.table(full_table_name).count()
    expected = expected_counts[table_name]

    if actual == expected:
        status = "OK"
    else:
        status = "MISMATCH"

    print(status, full_table_name, actual, "/", expected)
    total_rows = total_rows + actual

print("total:", total_rows)

## DESCRIBE DETAIL ve DESCRIBE HISTORY

Amaç: Bronze tablolarının gerçekten Delta olduğunu görmek ve Delta'nın tuttuğu version log'una bakmak. Örnek tablo: `trans`.

`DESCRIBE DETAIL` çıktısı: `format` = delta olmalı. `numFiles` ve `sizeInBytes` tablonun kaç Parquet dosyasından oluştuğunu ve diskteki boyutunu gösterir (CSV'deki ~69 MB'tan küçük çıkması beklenir, çünkü Parquet sıkıştırılmış bir format).

`DESCRIBE HISTORY` çıktısı: Her yazım bir `version`'dır. `operation` = CREATE OR REPLACE TABLE AS SELECT, `operationMetrics` içindeki `numOutputRows` = 1056320 olmalı.

In [0]:
%sql
DESCRIBE DETAIL workspace.bronze.trans

In [0]:
%sql
DESCRIBE HISTORY workspace.bronze.trans